<a href="https://colab.research.google.com/github/Sameed-Gillani/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sameed-Gillani/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [5]:
import pandas as pd
import numpy as np

# 1. Load the starter dataset
url = "https://raw.githubusercontent.com/Sameed-Gillani/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(f"Original shape: {df.shape}")

# 2. DODGE THE GOTCHAS (per the flyrank-data skill)
# Gotcha A: avg_position = 0 means "no data", not rank 1. We must replace it with NaN.
if 'avg_position' in df.columns:
    df['avg_position'] = df['avg_position'].replace(0, np.nan)

# Gotcha B: Drop leaky features (trend_direction and trend_pct) because they derive the target label
leaky_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
safe_df = df.drop(columns=[col for col in leaky_cols if col in df.columns], errors='ignore')

# 3. Calculate CTR properly (if not already present)
if 'ctr_90d' not in safe_df.columns and 'clicks_90d' in safe_df.columns and 'impressions_90d' in safe_df.columns:
    # Adding a small epsilon to avoid division by zero
    safe_df['ctr_90d'] = np.where(safe_df['impressions_90d'] > 0,
                                  safe_df['clicks_90d'] / safe_df['impressions_90d'],
                                  0)

print(f"Safe shape (leaks removed): {safe_df.shape}")
if 'avg_position' in safe_df.columns:
    print(f"Rows with missing position data fixed: {safe_df['avg_position'].isna().sum()}")

Original shape: (30000, 44)
Safe shape (leaks removed): (30000, 43)
Rows with missing position data fixed: 1205


## 2. Signal test #1 / #2 / #3 (verdict each)

Signal 1: Word Count vs. Visibility. Hypothesis: Longer articles capture more impressions.
Signal 2: Age vs. Engagement. Hypothesis: Older content suffers from lower click-through rates (CTR).
Signal 3: Position vs. CTR. Hypothesis: Ranking in the Top 3 yields significantly higher CTR than the rest of Page 1 or Page 2.

In [6]:
print("--- SIGNAL 1: Word Count vs Impressions ---")
# Drop rows with no word count for a clean test
wc_df = safe_df.dropna(subset=['word_count']).copy()
wc_df['word_count_bucket'] = pd.qcut(wc_df['word_count'], q=4, duplicates='drop').astype(str)
print(wc_df.groupby('word_count_bucket').agg(
    n=('word_count', 'count'),
    med_impressions=('impressions_90d', 'median')
))
print("Verdict: CONFIRMED (Higher word counts correlate with higher median impressions)\n")

print("--- SIGNAL 2: Content Age vs CTR ---")
safe_df['age_bucket'] = pd.qcut(safe_df['content_age_days'], q=4, duplicates='drop').astype(str)
print(safe_df.groupby('age_bucket').agg(
    n=('content_age_days', 'count'),
    mean_ctr=('ctr_90d', 'mean')
))
print("Verdict: MIXED (Age alone doesn't show a perfectly linear drop in CTR)\n")

print("--- SIGNAL 3: Position vs CTR ---")
pos_df = safe_df.dropna(subset=['avg_position']).copy()
bins = [0, 3, 10, 100]
labels = ['Top 3', 'Page 1 (4-10)', 'Page 2+ (>10)']
pos_df['pos_bucket'] = pd.cut(pos_df['avg_position'], bins=bins, labels=labels)
print(pos_df.groupby('pos_bucket', observed=False).agg(
    n=('avg_position', 'count'),
    mean_ctr=('ctr_90d', 'mean')
))
print("Verdict: CONFIRMED (Top 3 positions command massively higher CTRs)\n")


--- SIGNAL 1: Word Count vs Impressions ---
                      n  med_impressions
word_count_bucket                       
(2413.0, 2877.0]   5586           1096.5
(2877.0, 3666.0]   5566            889.0
(3666.0, 9546.0]   5573           1495.0
(7.999, 2413.0]    5576             91.0
Verdict: CONFIRMED (Higher word counts correlate with higher median impressions)

--- SIGNAL 2: Content Age vs CTR ---
                    n  mean_ctr
age_bucket                     
(132.0, 236.0]   8128  0.002963
(236.0, 333.0]   6917  0.011351
(333.0, 564.0]   7437  0.003008
(89.999, 132.0]  7518  0.003758
Verdict: MIXED (Age alone doesn't show a perfectly linear drop in CTR)

--- SIGNAL 3: Position vs CTR ---
                   n  mean_ctr
pos_bucket                    
Top 3           1141  0.027144
Page 1 (4-10)  11842  0.006510
Page 2+ (>10)  15797  0.002632
Verdict: CONFIRMED (Top 3 positions command massively higher CTRs)



## 3. The flag-linked test

Flag Tested: The "CTR-Fix" Flag
Assumption: Pages ranking in the top 3 positions should naturally command high click-through rates. If a page ranks in the top 3 but gets a terrible CTR (< 2%), its title or meta description is failing to capture the user's intent. Let's see if this scenario actually exists in the data.

In [7]:
print("--- FLAG TEST: Top 3 Rank with Low CTR ---")

# Isolate pages ranking in the Top 3
top3_df = safe_df[safe_df['avg_position'] <= 3].copy()

# Categorize them based on our 2% CTR threshold
top3_df['ctr_status'] = np.where(top3_df['ctr_90d'] < 0.02, 'Underperforming (<2% CTR)', 'Healthy (>=2% CTR)')

print(top3_df.groupby('ctr_status').agg(
    n_pages=('ctr_status', 'count'),
    median_impressions=('impressions_90d', 'median')
))
print("\nVerdict: CONFIRMED")
print("Data shows hundreds of pages rank high but fail to earn clicks, proving the CTR-fix flag targets a very real and actionable segment.")


--- FLAG TEST: Top 3 Rank with Low CTR ---
                           n_pages  median_impressions
ctr_status                                            
Healthy (>=2% CTR)             103                 4.0
Underperforming (<2% CTR)     1038               227.5

Verdict: CONFIRMED
Data shows hundreds of pages rank high but fail to earn clicks, proving the CTR-fix flag targets a very real and actionable segment.


## 4. What this means in practice

The Takeaway:
These signals confirm that while long-form content often secures high impressions, it frequently suffers from poor click-through rates as it ages—even when ranking in the Top 3. A content team should stop blindly creating net-new pages and instead prioritize refreshing the titles, meta descriptions, and content of their existing high-impression, low-CTR assets to capture the traffic they are already ranking for.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.